# ROM-FlameBench: a simple walkthrough

This notebook runs three complete experiments and one hyperparameter search:

1. Prepare and load the data.
2. Build the datasets for the compressor and the forecaster.
3. **POD-ARX**: linear compression and a linear forecaster.
4. **POD-LSTM**: the same compression and a recurrent forecaster trained on multi-step rollouts.
5. **CAE-ARX**: a convolutional autoencoder and a linear forecaster.
6. **HPO of POD-ARX**: tune the forecaster and the history lengths on validation data.
7. Compare the results.

Run the cells in order, from the repository root, using the project's Python environment.
The fits use the real dataset and can take time; the CAE needs a GPU to be quick.

**Confirmed:** snapshot zero is the initial steady state.
Physical cell volumes come from `grid.vtu`, so the test metrics include the integrated heat
release and its gain and phase errors.

## 1. Data preparation

Raw archives contain `data` with shape **(cell, field, time)**. Preparation maps the mesh cells
to pixels and writes float32 `.npy` images with shape **(time, field, 206, 104)**. Pixels
without a cell are invalid: a mask marks the valid ones, and every error below uses valid
pixels only. The images are read from disk in small batches. Existing prepared files are reused.

The supplied `phi` files contain the dimensionless forcing: $U(t)=U_{base}\phi(t)$.
The metadata defines filenames, field order, units and the sampling interval.

In [1]:
import time
from pathlib import Path

import numpy as np
import torch
from DataProcessing.loading import make_loader

from DataProcessing.prepare import prepare
from DataProcessing.metadata import load_metadata
from DataProcessing.Dataset import Dataset, CompressorDataset, ForecasterDataset
from utils import seed_everything

device = "cuda" if torch.cuda.is_available() else "cpu"
seed = 42
seed_everything(seed)
cpu_threads = 4
torch.set_num_threads(cpu_threads)
loader_options = dict(
    num_workers=0,                # Data is in RAM (Dataset.in_memory); workers only add startup time.
    pin_memory=device == "cuda",  # Useful for CPU-to-GPU transfers.
    persistent_workers=True,      # Reuse workers across training epochs.
    prefetch_factor=1,            # Batches queued per worker; image windows are large.
    multiprocessing_context="spawn",
)
pod_backend = "sklearn"           # Legacy default; "torch" fits on device (including CUDA).


def loader(dataset, batch_size=64, shuffle=False):
    return make_loader(dataset, batch_size, shuffle=shuffle, **loader_options)

Dataset.in_memory = True                     # Read the images into RAM once instead of from the shared disk.
torch.set_float32_matmul_precision("high")   # TF32 matrix products on the GPU.
torch.backends.cudnn.benchmark = True        # Fastest CAE convolutions; results are no longer bit-identical.
print(device, torch.cuda.get_device_name() if device == "cuda" else "")

metadata_path = Path("Data/metadata.json")
raw_directory = Path("Data/Raw")
output = Path("Experiments/Results") / time.strftime("walkthrough_%Y%m%dT%H%M%S")
print("Output directory:", output)

Nx = 9
Ni = 4
horizon = 10
K_eval = 50
rank = 16
batch_size = 256     # Batch size for the scaler, POD, ARX and evaluation passes.
validation_fraction = 0.2
blocks = 20
lstm_epochs = 100
cae_epochs = 20
cae_channels = [16, 32, 64]
trials = 2
hpo_fixed_dataset = {}

logging_config = {"logging": {"wandb": {"mode": "offline", "entity": "FireMark"}}}
output.mkdir(parents=True, exist_ok=True)

cuda NVIDIA L40S
Output directory: Experiments/Results/walkthrough_20260922T184047


In [2]:
prepare(metadata_path, raw_directory)
metadata = load_metadata(metadata_path)

print("Fields:", metadata["fields"])
print("Sampling interval:", metadata["dt"], "seconds")
for case in metadata["cases"]:
    print(case["split"], case["name"])

Keeping /srv/mlg/shared/FlameBench/Data/Images/Training/sineSweep_f1_f80_A02.npy
Keeping /srv/mlg/shared/FlameBench/Data/Images/Training/phi_sineSweep_f1_f80_A02.npy
Keeping /srv/mlg/shared/FlameBench/Data/Images/Training/sineSweep_f1_f80_A04.npy
Keeping /srv/mlg/shared/FlameBench/Data/Images/Training/phi_sineSweep_f1_f80_A04.npy
Keeping /srv/mlg/shared/FlameBench/Data/Images/Test/sine_f10_A03.npy
Keeping /srv/mlg/shared/FlameBench/Data/Images/Test/phi_sine_f10_A03.npy
Keeping /srv/mlg/shared/FlameBench/Data/Images/Test/sine_f10_A05.npy
Keeping /srv/mlg/shared/FlameBench/Data/Images/Test/phi_sine_f10_A05.npy
Keeping /srv/mlg/shared/FlameBench/Data/Images/Test/sine_f40_A03.npy
Keeping /srv/mlg/shared/FlameBench/Data/Images/Test/phi_sine_f40_A03.npy
Keeping /srv/mlg/shared/FlameBench/Data/Images/Test/sine_f40_A05.npy
Keeping /srv/mlg/shared/FlameBench/Data/Images/Test/phi_sine_f40_A05.npy
Keeping /srv/mlg/shared/FlameBench/Data/Images/Test/step_A03.npy
Keeping /srv/mlg/shared/FlameBench/

## 2. Datasets

Each training sweep is cut in time into **`blocks` equal blocks**. A fraction
`validation_fraction` of the blocks, evenly spaced, is validation; the others are training.
Both partitions therefore cover the whole sweep, low and high frequencies. Consecutive blocks
form one segment, and samples never cross a segment edge, so no frame is used by both
partitions. Test simulations are kept separate.

There are two kinds of dataset, and both use exactly the same split:

- A `CompressorDataset` sample is **one frame**.
- A `ForecasterDataset` sample is a **window** of `max(Nx, Ni) + 1` rows ending at time $t$.
  Row $s$ holds the state $x(s)$ and the forcing deviation $\phi(s+1)-1$, so the last row pairs
  $x(t)$ with the known next forcing. **Nx and Ni are independent**: states older than the last
  `Nx + 1` rows and forcing older than the last `Ni + 1` rows are set to zero. The targets are
  the next `horizon` states $x(t+1),\ldots,x(t+K)$.

The `Dataset` defines samples; a `DataLoader` groups them into batches.

`loader_options` applies to scaler, compressor, forecaster and validation loaders, including HPO.
`num_workers=0` disables multiprocessing. Start with 2 workers and prefetch 1 for images;
latent-only training may be faster with 0 workers. More workers are not always faster.
Workers read memory-mapped files; `Dataset.in_memory=True` caches full trajectories only in the
notebook process, avoiding a separate full-data copy in each spawned worker. Keep the data on
server-local storage when possible. Docker needs enough `/dev/shm` for queued batches.

Progress uses plain-text tqdm, so no notebook widget extension is needed. Restart the kernel
after pulling these changes. SVD shows a start/end stage bar: the library provides no iteration callback.


In [3]:
split = dict(validation_fraction=validation_fraction, blocks=blocks)
train_compressor_ds = CompressorDataset(metadata, "train", **split)
train_forecaster_ds = ForecasterDataset(metadata, "train", Nx=Nx, Ni=Ni, **split)
val_compressor_ds = CompressorDataset(metadata, "validation", **split)
val_forecaster_ds = ForecasterDataset(metadata, "validation", Nx=Nx, Ni=Ni, **split)

test_ds = ForecasterDataset(metadata, "test", Nx=Nx, Ni=Ni)

sample = train_forecaster_ds[0]
print("Training segments:", [(case["name"], start, stop) for case, start, stop in train_compressor_ds.segments])
print("Training frames:", len(train_compressor_ds))
print("Training windows:", len(train_forecaster_ds))
print("States:", tuple(sample["states"].shape))  # (max(Nx, Ni) + 1, fields, height, width)
print("Forcing:", sample["forcing"].tolist())   # phi(s+1) - 1 for each row
print("Target:", tuple(sample["target"].shape))  # (horizon, fields, height, width)

Training segments: [('sineSweep_f1_f80_A02', 0, 400), ('sineSweep_f1_f80_A02', 600, 1400), ('sineSweep_f1_f80_A02', 1600, 2400), ('sineSweep_f1_f80_A02', 2600, 3400), ('sineSweep_f1_f80_A02', 3600, 4001), ('sineSweep_f1_f80_A04', 0, 400), ('sineSweep_f1_f80_A04', 600, 1400), ('sineSweep_f1_f80_A04', 1600, 2400), ('sineSweep_f1_f80_A04', 2600, 3400), ('sineSweep_f1_f80_A04', 3600, 4001)]
Training frames: 6402
Training windows: 6302
States: (10, 11, 206, 104)
Forcing: [0.0, 0.0, 0.0, 0.0, 0.0, 0.003794550895690918, 0.004431724548339844, 0.005070328712463379, 0.005710244178771973, 0.006351470947265625]
Target: (1, 11, 206, 104)


## 3. POD-ARX

The fields have different units and scales. First, we estimate a mean and standard deviation
for each field **using training frames only and excluding empty pixels**. The scaler is fitted
once and shared by every experiment; the datasets then return scaled frames.

**POD** flattens each scaled image into the vector of its valid pixels and keeps the first
`rank` modes of the training frames (legacy randomized SVD: oversampling 20, 7 iterations,
seed 42). `encode` maps a frame to `rank` coefficients and `decode` maps them back. The
reconstruction error measures compression alone.

Keep `pod_backend="sklearn"` for the legacy implementation. Set `pod_backend="torch"` to use
`torch.svd_lowrank` on `device`, with rank + 20 sampled directions (capped by matrix dimensions),
7 iterations and seed 42. This is the same centered randomized-POD approach, but the basis is
not numerically identical to sklearn. Compare reconstruction error and wall time before choosing it.
Only the SVD fit moves to GPU; encoding and decoding still use NumPy on CPU.
The current training matrix is roughly 6 GB in float32, plus SVD workspace and CPU copies;
use the L40S first. No GPU speedup has been measured locally.


In [4]:
from DataProcessing.scaling import FeatureScaler
from Baselines.OrderReduction.Linear.POD import POD
from Experiments.evaluation import reconstruction_error

scaler = FeatureScaler(train_compressor_ds.mask).fit(loader(train_compressor_ds, batch_size=batch_size))
train_compressor_ds = CompressorDataset(metadata, "train", scaler=scaler, **split)
val_compressor_ds = CompressorDataset(metadata, "validation", scaler=scaler, **split)

pod = POD(rank=rank, batch_size=batch_size, backend=pod_backend, device=device).fit(
    train_compressor_ds, loader_options=loader_options)
print("POD validation reconstruction MSE:", reconstruction_error(pod, val_compressor_ds, loader_options=loader_options))

Reconstruction error: 100%|██████████| 100/100 [00:08<00:00, 12.00it/s]

POD validation reconstruction MSE: 0.017777484032602833


Given a compressor, a `ForecasterDataset` encodes its frames once and keeps the small latent
trajectories in memory. The helpers below are shared by all experiments:

- `latent_windows` builds the training windows and the validation windows (`K_eval` targets).
- `validation_mse` is the **selection objective**: from each validation window, forecast
  `K_eval` steps recursively (each prediction is fed back), decode, and compare with the scaled
  frames on valid pixels. Windows are `K_eval` apart, so each frame is compared once.
- `test_rollouts` forecasts each test simulation from its steady first snapshot to the end.

**ARX** is a ridge regression of the increment $z_{t+1}-z_t$ on the latent history, the forcing
history and a constant:

$$\widehat z_{t+1} = z_t + W\,[z_{t-N_x},\ldots,z_t,\ \phi_{t+1-N_i}-1,\ldots,\phi_{t+1}-1,\ 1].$$

`alpha` controls the ridge penalty. ARX has a closed-form fit on one-step targets.

In [ ]:
from Baselines.Forecast.Classical.ARX import ARX
from Experiments.evaluation import validation_error, evaluate


scores, tests = {}, {}
train_forecaster_dataset = ForecasterDataset(metadata, "train", horizon=1, Nx=Nx, Ni=Ni, scaler=scaler,compressor=pod, **split)
val_forecaster_dataset = ForecasterDataset(metadata, "validation",horizon=K_eval,stride=K_eval, Nx=Nx, Ni=Ni, scaler=scaler, **split)

arx = ARX(Nx=Nx, Ni=Ni, alpha=1e-4).fit(loader(train_forecaster_dataset, batch_size=batch_size))
scores["pod_arx"] = validation_error(arx, pod, val_forecaster_dataset, loader_options=loader_options)
tests["pod_arx"] = evaluate(arx, test_ds, pod, scaler, output / "pod_arx")
print("POD-ARX validation field MSE:", scores["pod_arx"])

Test sine_f10_A05:  83%|████████▎ | 1665/2000 [01:57<00:25, 13.14it/s]

Each test rollout starts from **snapshot zero**, repeated to fill the history. No later
ground-truth field is supplied to the model. Forcing before time zero is the unforced value
($\phi=1$). At each step the forecaster predicts the next latent state, the compressor decodes
it, and the prediction is fed back as the newest state.

For each field, NRMSE is the RMSE divided by the ground-truth standard deviation, pooling all
evaluated cells and times in that case. The mean NRMSE averages the 11 field scores. Snapshot
zero is excluded because it was given to the model.

## 4. POD-LSTM

The **LSTM** reads the same rows $[z_s,\ \phi_{s+1}-1]$ and predicts the increment of the last
state. It is trained on **multi-step rollouts**: each training window has `horizon` targets,
the prediction of each step is fed back, and the loss is

$$(1-w)\,L_1 + w\,\tfrac{1}{K}\textstyle\sum_{k=1}^{K} L_k .$$

Early stopping uses the same loss on the validation windows, which have `K_eval` targets, so
the model is trained on $K$ steps and checked on longer forecasts.

In [ ]:
from Baselines.Forecast.DL.networks import LSTM

train_forecaster_dataset = ForecasterDataset(metadata, "train", horizon=horizon, Nx=Nx, Ni=Ni, scaler=scaler,compressor=pod, **split)
val_forecaster_dataset = ForecasterDataset(metadata, "validation",horizon=K_eval,stride=K_eval, Nx=Nx, Ni=Ni, scaler=scaler, **split)
lstm = LSTM(input_size=pod.rank + 1, output_size=pod.rank, Nx=Nx, Ni=Ni, hiddens=[64], normalization="layer", input_normalization="batch", epochs=lstm_epochs, patience=20, device=device)
lstm.fit(loader(train_forecaster_dataset, batch_size=batch_size, shuffle=True), loader(ForecasterDataset(metadata, "validation",horizon=K_eval,stride=K_eval, Nx=Nx, Ni=Ni, scaler=scaler,compressor=pod, **split), batch_size=batch_size))
scores["pod_lstm"] = validation_error(lstm, pod, val_forecaster_dataset, loader_options=loader_options)
tests["pod_lstm"] = evaluate(lstm, test_ds, pod, scaler, output / "pod_lstm")
print("POD-LSTM validation field MSE:", scores["pod_lstm"])

## 5. CAE-ARX

The **CAE** compresses each image with convolutions (`channels` gives the width of each
level), then a linear layer to `rank` values. The decoder mirrors the encoder, so the output
has exactly the input shape. Training shuffles the training frames every epoch and keeps the
epoch with the lowest validation reconstruction error. The ARX forecaster is the same as in
section 3, now on the CAE latents.

In [ ]:
from Baselines.OrderReduction.DL.CAE import CAE

cae = CAE(rank=rank, channels=cae_channels, epochs=cae_epochs, batch_size=16, device=device)
cae.fit(train_compressor_ds, validation=val_compressor_ds, loader_options=loader_options)
print("CAE validation reconstruction MSE:", reconstruction_error(cae, val_compressor_ds, loader_options=loader_options))

train_forecaster_dataset = ForecasterDataset(metadata, "train", horizon=1, Nx=Nx, Ni=Ni, scaler=scaler,compressor=cae, **split)
val_forecaster_dataset = ForecasterDataset(metadata, "validation",horizon=K_eval,stride=K_eval, Nx=Nx, Ni=Ni, scaler=scaler, **split)
cae_arx = ARX(Nx=Nx, Ni=Ni, alpha=1e-4).fit(loader(train_forecaster_dataset, batch_size=batch_size))
scores["cae_arx"] =validation_error(cae_arx, cae, val_forecaster_dataset, loader_options=loader_options)
tests["cae_arx"] = evaluate(cae_arx, test_ds, cae, scaler, output / "cae_arx")
print("CAE-ARX validation field MSE:", scores["cae_arx"])

## 6. HPO of POD-ARX

`Experiments.HPO.optimize` receives only the classes and a config. Every class lists its search
space in `hyperparameters_ranges`, and `build` creates an object from chosen values. Values
written in the config are **fixed**; the other entries of the ranges are **tuned** with Optuna.
Here the POD rank is fixed, and the search tunes ARX's `alpha` together with the dataset's `Nx`
and `Ni`. ARX fixes the dataset `horizon` to 1, because its closed-form fit is one-step.

With a compressor and a forecaster the search runs in stages, each keeping the best result of
the stage before:

1. the compressor, on the validation reconstruction error (POD has nothing to tune: one fit);
2. the forecaster and the dataset, with the compressor frozen, on the validation field MSE;
3. joint training of a neural forecaster and an autoencoder (not used by POD-ARX).

`optimize` returns the best config. `fit` then trains it from scratch, and `test` evaluates it.

In [ ]:
from Experiments.HPO import optimize
from Experiments.run import fit, test

hpo_config = {
    "metadata": str(metadata_path), "output": str(output), "run_name": "hpo_pod_arx", "seed": seed,
    "device": device, "validation_fraction": validation_fraction, "blocks": blocks, "K_eval": K_eval,
    "trials": trials, "batch_size": batch_size, "preprocessing_batch_size": batch_size, "cpu_threads": cpu_threads,
    "dataloader": loader_options,
    "compressor": {"name": "pod", "rank": rank, "backend": pod_backend},
    "dataset": dict(hpo_fixed_dataset),
    "forecaster": {"name": "arx"},
    "evaluation": {"heat_release": True},
    **logging_config,
}
best = optimize(hpo_config, POD, ARX)
print("Best config:", best)

best_config = {**hpo_config, **best}
scores["hpo_pod_arx"] = fit(best_config)
tests["hpo_pod_arx"] = test(best_config)

## 7. Results

The validation field MSE is the selection objective (scaled fields, valid pixels, `K_eval`-step
forecasts). Each test simulation is forecast from its first snapshot to the end, and scored in
physical units:

- **NRMSE** of each field: RMSE divided by the standard deviation of the reference field; the
  mean averages the 11 fields.
- **SSIM** of each field: structural similarity of the predicted and reference images, averaged
  over valid pixels and frames; 1 is a perfect match.
- **Q L2**: relative L2 error of the heat release integrated over the cell volumes.
- **Gain** and **phase** errors of the heat-release response to the forcing, at the forcing
  frequency, from 0.5 s on. Only sine cases have them.
- **ms/step**: inference time of one step (latent transition, decoding and inverse scaling).


In [ ]:
def show(value, digits=4):
    return "-" if value is None else f"{value:.{digits}g}"

fields = metadata["fields"]
for name, results in tests.items():
    print(f"\n{name}: validation MSE {show(scores[name])}")
    print(f"{'case':<14}{'NRMSE':>9}{'SSIM':>9}{'Q L2':>9}{'gain err':>10}{'phase err':>11}{'ms/step':>9}")
    for case, m in results.items():
        gain_phase = m.get("gain_phase", {})
        print(f"{case:<14}{show(m['mean_nrmse']):>9}{show(m['mean_ssim']):>9}"
              f"{show(m.get('heat_release_relative_l2')):>9}{show(gain_phase.get('relative_gain_error')):>10}"
              f"{show(gain_phase.get('phase_error_deg'), 3):>11}{show(1000 * m['seconds_per_step'], 3):>9}")
    for metric in ("field_nrmse", "field_ssim"):
        print(f"\n{metric:<14}" + "".join(f"{field:>8}" for field in fields))
        for case, m in results.items():
            print(f"{case:<14}" + "".join(f"{show(m[metric][field], 3):>8}" for field in fields))

Finally, send the scalar results to **TensorBoard and W&B**. W&B defaults to offline here; the
project's `.env` can set `WANDB_MODE=online` and provide the API key for `FireMark`.
Credentials are not saved with these settings. TensorBoard can read the results with
`tensorboard --logdir Experiments/Results`.

In [ ]:
from Experiments.logging import ExperimentLogger

logger = ExperimentLogger(output / "summary", {"run_name": output.name, "seed": seed, **logging_config})
try:
    for step, (name, results) in enumerate(tests.items()):
        values = {f"{name}/validation_field_mse": scores[name]}
        for case, metrics in results.items():
            values[f"{name}/test/{case}/mean_nrmse"] = metrics["mean_nrmse"]
            values[f"{name}/test/{case}/mean_ssim"] = metrics["mean_ssim"]
        logger.log(values, step=step)
finally:
    logger.close()